In [53]:
import pandas as pd
import glob
import json
from tqdm import tqdm
import re, unicodedata
from rapidfuzz import fuzz

In [38]:
report_paths = glob.glob('./Result/analysis_reports/*.json')

In [64]:
def normalize(txt: str) -> str:
    txt = unicodedata.normalize('NFKD', txt)          # 兼容全角/半角
    txt = re.sub(r'\s+', ' ', txt)                    # 合并空白
    txt = re.sub(r'[^\w\s]', '', txt)                 # 去标点（视情况保留 %、$ 等）
    return txt.lower().strip()

def section_similarity(snippet_norm, section_norm):
    # token_set_ratio 对长文本更稳健
    return fuzz.token_set_ratio(snippet_norm, section_norm)

split_tag = "********************Item 7: Management's Discussion and Analysis of Financial Condition and Results of Operations********************"

def detect_report_label(snippet, ticker, date, thresh=90, margin=0.1):
    with open(f'./Data/10-K/item_1a_7/{ticker}_{date}.txt', 'r') as f:
        item_1a_7 = f.read()
    item_1a, item_7 = item_1a_7.split(split_tag, maxsplit=1)

    # 预处理
    snippet_norm = normalize(snippet)
    item1a_norm  = normalize(item_1a)
    item7_norm   = normalize(item_7)

    # 相似度
    score_1a = fuzz.token_set_ratio(snippet_norm, item1a_norm)
    score_7  = fuzz.token_set_ratio(snippet_norm, item7_norm)

    # 判定逻辑
    if score_1a >= thresh and score_1a - score_7 >= margin:
        return 'item_1a'
    elif score_7 >= thresh and score_7 - score_1a >= margin:
        return 'item_7'
    elif score_1a >= thresh and score_7 >= thresh and score_1a == 100 and score_7 == 100:
        return 'item_1a, item_7'  # 特例：两者相似度相同，可能是模糊匹配或内容重叠
    else:
        raise ValueError(
            f"无法明确归属（可能相似度不足）\n"
            f"Ticker={ticker}, Date={date}\n"
            f"score_1a={score_1a:.1f}, score_7={score_7:.1f}\n"
            f"Snippet={snippet[:120]}..."
        )

In [65]:
for r_p in tqdm(report_paths):
    file_name = r_p.split("/")[-1]
    ticker, date = file_name.split("_")[:2]
    save_path = r_p.replace("/analysis_reports/", "/analysis_reports_labeled/")

    with open(r_p, 'r') as f:
        report = json.load(f)[0]

    for i, ev in enumerate(report.get("evidence_list", [])):
        item_label = ev.get("item_label", detect_report_label(ev["snippet"], ticker, date))
        new_ev = {
            "snippet":           ev.get("snippet", ""),
            "item_label":        item_label,
            "score_calculation": ev.get("score_calculation", ""),
            "reasoning_summary": ev.get("reasoning_summary", "")
        }
        report["evidence_list"][i] = new_ev

    with open(save_path, "w", encoding="utf-8") as f:
        json.dump([report], f, indent=4, ensure_ascii=False)

100%|██████████| 1728/1728 [01:41<00:00, 17.01it/s]
